In [1]:
import asyncio
import sys
import threading
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def run_task1_scans():
    server_script = str(Path("recon_server.py").resolve())
    errlog_path = str(Path("task1_mcp_stderr.log").resolve())

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[server_script],
    )

    with open(errlog_path, "w", encoding="utf-8") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await asyncio.wait_for(session.initialize(), timeout=30)

                tools_resp = await session.list_tools()
                print("Discovered MCP tools:", [t.name for t in tools_resp.tools])

                calls = [
                    ("banner_grab", {"target": "8.8.8.8", "port": 80}),
                    ("banner_grab", {"target": "127.0.0.1", "port": 2121}),
                    ("anonymous_ftp_check", {"target": "127.0.0.1", "port": 2121}),
                    ("http_probe", {"target": "127.0.0.1", "port": 8080, "path": "/"}),
                ]

                for tool_name, arguments in calls:
                    result = await session.call_tool(tool_name, arguments)
                    print(f"\n{'=' * 70}")
                    print(f"TOOL: {tool_name} | ARGS: {arguments} | isError: {result.isError}")
                    print(f"{'=' * 70}")
                    for item in result.content:
                        text = getattr(item, "text", str(item))
                        print(text)


def run_in_new_loop(coro):
    box = {}

    def _worker():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(coro)
        except BaseException as exc:
            box["error"] = exc
        finally:
            loop.close()

    thread = threading.Thread(target=_worker)
    thread.start()
    thread.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


run_in_new_loop(run_task1_scans())

Discovered MCP tools: ['banner_grab', 'anonymous_ftp_check', 'http_probe', 'nmap_service_scan']

TOOL: banner_grab | ARGS: {'target': '8.8.8.8', 'port': 80} | isError: False
REFUSED: Target '8.8.8.8' is outside the authorized allowlist ['127.0.0.1'].

TOOL: banner_grab | ARGS: {'target': '127.0.0.1', 'port': 2121} | isError: False
target=127.0.0.1 port=2121 state=open banner='220 (vsFTPd 2.3.4)'

TOOL: anonymous_ftp_check | ARGS: {'target': '127.0.0.1', 'port': 2121} | isError: False
target=127.0.0.1 port=2121 anonymous_login=ALLOWED welcome='220 (vsFTPd 2.3.4)' response='230 Login successful.' sample_entries=[]

TOOL: http_probe | ARGS: {'target': '127.0.0.1', 'port': 8080, 'path': '/'} | isError: False
url=http://127.0.0.1:8080/ status=200 server=Apache/2.2.8 (Ubuntu) DAV/2 x_powered_by=PHP/5.2.4-2ubuntu5.10 body_preview="<html><head><title>Metasploitable2 - Linux</title></head><body>\n<pre>\n\n                _                  _       _ _        _     _      ____  \n _ __ ___   ___